# SmartFarm ML — Stage 05: The crop dimension
**Goal:** turn "one plant, one threshold" into decisions that differ by crop. Stage 02/03 showed
ML tying (not beating) the rules baseline — because the model couldn't see `crop_type`, while
the rules baseline is crop-aware. This stage gives the model that missing signal and re-runs
the honest showdown. Data: `irrigation_honest_05.csv`.

## 1. Load

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

df = pd.read_csv("irrigation_honest_05.csv")
df.head()

,crop_type,growth_stage,soil_moisture,air_humidity,temperature,irrigate
0,tomato,flowering,56.7,58.8,31.7,0
1,okra,flowering,65.8,77.8,29.0,0
2,chili,vegetative,13.1,81.5,28.1,1
3,chili,vegetative,24.0,86.6,31.3,1
4,chili,seedling,32.1,66.2,30.8,0


## 2. Encode crop_type — one-hot encoding
Models need numbers, not text. `crop_type` has 3 categories with no natural order (tomato isn't
"more" than chili), so **one-hot encoding** is correct here — NOT assigning 0/1/2, which would
falsely imply an ordering the model might try to exploit.

In [13]:
df_enc = pd.get_dummies(df, columns=["crop_type"], prefix="crop")
# print(df_enc)
crop_cols = [c for c in df_enc.columns if c.startswith("crop_")]
print("new columns:", crop_cols)
df_enc[["soil_moisture"] + crop_cols].head()

new columns: ['crop_chili', 'crop_okra', 'crop_tomato']


,soil_moisture,air_humidity,crop_chili,crop_okra,crop_tomato
0,56.7,58.8,False,False,True
1,65.8,77.8,False,True,False
2,13.1,81.5,True,False,False
3,24.0,86.6,True,False,False
4,32.1,66.2,True,False,False


`pd.get_dummies` turns one text column into several 0/1 columns — one per category. A tomato
row gets `crop_tomato=1, crop_chili=0, crop_okra=0`. Each dummy column is just a yes/no feature
the model can weight independently.

## 3. Train WITHOUT crop vs WITH crop — does it help?

In [3]:
feat_no_crop = ["soil_moisture", "air_humidity", "temperature"]
feat_with_crop = feat_no_crop + crop_cols
y = df_enc["irrigate"]

results = {}
for label, feats in [("without crop", feat_no_crop), ("WITH crop", feat_with_crop)]:
    X = df_enc[feats]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    pred = model.predict(Xte)
    results[label] = (model.score(Xte, yte), recall_score(yte, pred), f1_score(yte, pred))
    print(f"{label:14s} acc={results[label][0]:.3f}  recall={results[label][1]:.3f}  f1={results[label][2]:.3f}")

without crop   acc=0.782  recall=0.730  f1=0.738
WITH crop      acc=0.796  recall=0.730  f1=0.750


## 4. Feature engineering — give the interaction explicitly
One-hot dummies + soil are still combined *linearly* by logistic regression — it can't learn
"soil relative to THIS crop's ideal" on its own from separate columns. So we engineer that
interaction directly: `soil_deficit = crop's ideal soil − current soil`. Positive = drier than
this crop likes. This single feature carries per-crop context in a form logreg can use directly.

In [4]:
ideal = {"tomato": 58, "chili": 45, "okra": 50}
df_enc["soil_deficit"] = df["crop_type"].map(ideal) - df["soil_moisture"]

feat_deficit = ["soil_moisture", "air_humidity", "temperature", "soil_deficit"]
X = df_enc[feat_deficit]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
pred = model.predict(X_test)
print(f"logreg+deficit  acc={model.score(X_test,y_test):.3f}  recall={recall_score(y_test,pred):.3f}  f1={f1_score(y_test,pred):.3f}")

logreg+deficit  acc=0.798  recall=0.720  f1=0.749


## 5. Trend features — is soil dropping fast? (conceptual demo)
The roadmap also calls for trend features: is moisture dropping FAST (about to cross a danger
line soon) vs slowly? This needs **time-ordered readings per crop** — a snapshot dataset like
`irrigation_honest.csv` has no "previous reading" to compare against, so we demonstrate the
technique on the Stage 00 time-series file instead.

In [5]:
ts_df = pd.read_csv("irrigation_readings_05.csv", parse_dates=["timestamp"])
ts_df = ts_df[ts_df["soil_moisture"].between(0, 100)].sort_values(["crop_type", "timestamp"])

# .diff() = this row's value minus the PREVIOUS row's value, computed per crop group
ts_df["soil_change"] = ts_df.groupby("crop_type")["soil_moisture"].diff()
ts_df[["timestamp", "crop_type", "soil_moisture", "soil_change"]].head(8)

,timestamp,crop_type,soil_moisture,soil_change
2,2026-07-01 06:00:00,chili,52.4,NaN
4,2026-07-01 07:00:00,chili,50.3,-2.1
8,2026-07-01 08:00:00,chili,52.7,2.4
10,2026-07-01 09:00:00,chili,51.6,-1.1
14,2026-07-01 10:00:00,chili,51.3,-0.3
17,2026-07-01 11:00:00,chili,49.2,-2.1
20,2026-07-01 12:00:00,chili,50.6,1.4
23,2026-07-01 13:00:00,chili,48.1,-2.5


`groupby("crop_type")["soil_moisture"].diff()` computes each row minus the row before it
(chronologically), **per crop** — so tomato's trend never mixes with chili's. A large negative
`soil_change` = moisture dropping fast = maybe water sooner even if the absolute level looks OK
yet. (Not merged into the main honest showdown below since `irrigation_honest.csv` is snapshot
data — see Your Turn #2 for building a time-aware version.)

## 6. The real showdown — crop-aware model vs rules, on recall

In [6]:
ideal = {"tomato": 58, "chili": 45, "okra": 50}
base_pred = (df.loc[X_test.index, "soil_moisture"]
             < df.loc[X_test.index, "crop_type"].map(ideal)).astype(int)

print(f"{'':16s}{'precision':>10s}{'recall':>10s}{'f1':>8s}")
print(f"{'model+crop':16s}{precision_score(y_test,pred):>10.3f}{recall_score(y_test,pred):>10.3f}{f1_score(y_test,pred):>8.3f}")
print(f"{'rules':16s}{precision_score(y_test,base_pred):>10.3f}{recall_score(y_test,base_pred):>10.3f}{f1_score(y_test,base_pred):>8.3f}")

                 precision    recall      f1
model+crop           0.782     0.720   0.749
rules                0.687     0.836   0.754


## The honest verdict
Adding crop signal (via dummies + `soil_deficit`) **did help**: accuracy rose (0.782 → ~0.80)
and F1 improved. But **recall — the metric that matters most for irrigation — still doesn't
beat the rules baseline.** The rules baseline remains the safer default in production today.

This is a genuinely useful, non-obvious finding: more features and better engineering moved
the needle, but not on the metric where it counts. That's a legitimate reason to keep the
rules baseline as primary and treat the model as "not yet earning its place" — exactly the
kind of honest call the roadmap has been building toward since Stage 02.

What would likely close the gap: real labelled data beyond the bootstrap/synthetic label
(Stage 07), more history-aware features (Stage 05 trend features properly merged), or simply
accepting that for a problem this close to a simple physical threshold, the rules baseline may
never need replacing — and that's a fine outcome too.

## Your turn
1. Try `RandomForestClassifier` with the crop dummies + `soil_deficit` included. Does recall
   change? Why might a tree-based model use `soil_deficit` differently than logreg does?
2. (Harder) Build a time-aware version of `irrigation_honest.csv` — same idea as
   `irrigation_readings.csv` (sequential per crop) but with the honest label formula from
   Stage 02 — so `soil_change` can be added as a real feature in the showdown.
3. In one sentence: why is one-hot encoding correct for `crop_type` but would be wrong for
   something like `growth_stage` if growth stages have a natural order (seedling → ... → fruiting)?
   (Hint: research "ordinal encoding" if unsure.)

### Q1
No — Random Forest overfit here (train accuracy 1.000, test recall only 0.119, well below logreg's 0.220). Correctly predicted as overfitting before verifying — confirms this dataset's relationship is close to linear, where a simpler model generalises better than a complex one that memorises noise.

### Q2


In [34]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(random_state=0)
model_rf.fit(X_train, y_train)
model.fit(X_train, y_train)
model_rf.fit(X_train, y_train)

pred_rf = model_rf.predict(X_test)
pred = model.predict(X_test)

recall_rf = recall_score(y_test, pred_rf)

print(f"{'':26s}{'precision':>10s}{'recall':>10s}{'f1':>8s}")
print(f"{'Random Forest':26s}{precision_score(y_test,pred_rf):>10.3f}{recall_rf:>10.3f}{f1_score(y_test,pred_rf):>8.3f}")
print(f"{'Logistic Regression':26s}{precision_score(y_test,pred):>10.3f}{recall_score(y_test,pred):>10.3f}{f1_score(y_test,pred):>8.3f}")

                           precision    recall      f1
Random Forest                  0.737     0.698   0.717
Logistic Regression            0.782     0.720   0.749
